# 🏋️ 03 — Sarathi AI: QLoRA Fine-Tuning (Mistral 7B)

Fine-tunes Mistral 7B using QLoRA on a curated Indian chat dataset.

**Requirements:**
- Google Colab T4 GPU (free) or A100 (Colab Pro)
- Hugging Face account with Mistral 7B access accepted

In [ ]:
!nvidia-smi
!pip install -q transformers datasets accelerate peft bitsandbytes trl huggingface_hub

In [ ]:
from huggingface_hub import login
login()  # Enter HF token — get from https://huggingface.co/settings/tokens

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = 'mistralai/Mistral-7B-v0.1'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb, device_map='auto')
print('✅ Model loaded')

In [ ]:
from datasets import Dataset

SYSTEM = 'You are Sarathi AI, India-focused assistant by David at Nexuzy Lab, Kolkata. Answer in the user language (English/Hindi/Bengali).'

examples = [
    {'text': f'{SYSTEM}\nUser: What is IPC Section 302?\nSarathi: IPC Section 302 deals with punishment for murder — death penalty or life imprisonment plus fine.'},
    {'text': f'{SYSTEM}\nUser: Best budget hotel in Kolkata under 2000?\nSarathi: Broadway Hotel (₹800-₹2000, Ganesh Chandra Avenue) and Hotel Centrum (₹900-₹2200, Marquis Street) are excellent budget options.'},
    {'text': f'{SYSTEM}\nUser: What does Gita say about karma?\nSarathi: The Gita (Chapter 3) teaches perform your duty without attachment to results — Karmayoga.'},
    {'text': f'{SYSTEM}\nUser: How to reach Dakshineswar Temple from Howrah?\nSarathi: Take Circular Railway from Sealdah (~40 min, ₹5-10) or taxi/Uber (~30 min, ₹200-300).'},
    {'text': f'{SYSTEM}\nUser: Victoria Memorial timing and entry fee?\nSarathi: Open 10AM-5PM, closed Monday. Entry ₹30 for Indians, ₹500 for foreigners.'},
    {'text': f'{SYSTEM}\nUser: What is Article 21?\nSarathi: Article 21 of the Indian Constitution guarantees the Right to Life and Personal Liberty to all persons.'},
    {'text': f'{SYSTEM}\nUser: भारत का सबसे बड़ा शहर कौन सा है?\nSarathi: भारत का सबसे बड़ा शहर मुंबई है जनसंख्या के आधार पर, लेकिन दिल्ली NCR क्षेत्र में सबसे बड़ा है।'},
    {'text': f'{SYSTEM}\nUser: কলকাতার বিখ্যাত খাবার কী কী?\nSarathi: কলকাতার বিখ্যাত খাবার হল রসগোল্লা, কষা মাংস, ফুচকা, কাঠি রোল, এবং মিষ্টি দই।'},
]
dataset = Dataset.from_list(examples)
print(f'Dataset: {len(dataset)} samples')

In [ ]:
tokenized = dataset.map(lambda x: tokenizer(x['text'], truncation=True, padding='max_length', max_length=256), batched=True)
tokenized = tokenized.remove_columns(['text'])
print('Tokenized dataset ready')

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=8, lora_alpha=16, target_modules=['q_proj','k_proj','v_proj','o_proj'], lora_dropout=0.05, bias='none', task_type='CAUSAL_LM')
model = get_peft_model(model, lora)
model.print_trainable_parameters()

args = TrainingArguments(output_dir='sarathi-qlora', num_train_epochs=3, per_device_train_batch_size=2,
    gradient_accumulation_steps=4, learning_rate=2e-4, fp16=True, logging_steps=1, report_to='none')

trainer = Trainer(model=model, args=args, train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False))
trainer.train()
print('✅ Training complete!')

In [ ]:
model.save_pretrained('sarathi-qlora-model')
tokenizer.save_pretrained('sarathi-qlora-model')
print('Saved to sarathi-qlora-model')
# Push to HuggingFace (optional)
# model.push_to_hub('david0154/Sarathi-AI')
# tokenizer.push_to_hub('david0154/Sarathi-AI')